In [1]:
# Differentible observable of orbit

In [2]:
# problem-0: jit/vmap ordering
# problem-1: optimize performance for one initial condition
# problem-2: optimize vmap over initials
# problem-3: optimize nested derivatives

In [3]:
from typing import Any
from typing import Callable

import jax
from jax import Array

def mapping(x:Array, k:Array) -> Array:
    qx, qy, px, py = x
    cx, sx, cy, sy, mu = k
    Qx = cx*qx + sx*(px + qx**2 - qy**2 + mu*(qx**3 - 3*qx*qy**2))
    Qy = cy*qy + sy*(py - 2*qx*qy + mu*(-3*qx**2*qy + qy**3))
    Px = cx*(px + qx**2 - qy**2 + mu*(qx**3 - 3*qx*qy**2)) - sx*qx
    Py = cy*(py - 2*qx*qy + mu*(-3*qx**2*qy + qy**3)) - sy*qy
    return jax.numpy.stack([Qx, Qy, Px, Py])


def exponential(length:int, degree:float=1.0) -> Array:
    t = jax.numpy.linspace(0.0, (length - 1.0)/length, length)
    w = jax.numpy.exp(-1.0/((1.0 - t)**degree*t**degree))
    return w/w.sum()


def frequency(weights: Array, mapping: Callable[..., Array]) -> Callable[..., Array]:
    factor = 2.0 * jax.numpy.pi
    def closure(state: Array, *args: Any) -> Array:
        qs, ps = jax.numpy.reshape(state, (2, -1))
        initial = jax.numpy.arctan2(qs, ps)
        total = jax.numpy.zeros_like(initial)
        def scan_body(carry: tuple[Array, Array, Array], weight: Array) -> tuple[tuple[Array, Array, Array], None]:
            state, initial, total = carry
            state = mapping(state, *args)
            qs, ps = jax.numpy.reshape(state, (2, -1))
            current = jax.numpy.arctan2(qs, ps)
            delta = (current - initial) % factor
            total = total + weight * delta
            return (state, current, total), None
        (_, _, total), _ = jax.lax.scan(scan_body, (state, initial, total), weights)
        return total/factor
    return closure

In [4]:
# Set data type

jax.config.update("jax_enable_x64", True)

In [5]:
# Set device

device, *_ = jax.devices('cpu')
jax.config.update('jax_default_device', device)

In [6]:
# Set map parameters

nux, nuy = 0.168, 0.201
mux, muy = 2*jax.numpy.pi*nux, 2*jax.numpy.pi*nuy
cx, sx, cy, sy = jax.numpy.cos(mux), jax.numpy.sin(mux), jax.numpy.cos(muy), jax.numpy.sin(muy)
mu = 0.0

In [7]:
# Test map

k = jax.numpy.asarray([cx, sx, cy, sy, mu])
x = jax.numpy.array([0.0, 0.0, 0.0, 0.0])

print(mapping(x, k))
print(jax.jacrev(mapping, 0)(x, k))
print(jax.jacrev(mapping, 1)(x, k))

[0. 0. 0. 0.]
[[ 0.49272734  0.          0.87018375  0.        ]
 [ 0.          0.30303527  0.          0.95297934]
 [-0.87018375  0.          0.49272734  0.        ]
 [ 0.         -0.95297934  0.          0.30303527]]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [8]:
# Test frequency function

ws = exponential(2**10)
fn = jax.jit(frequency(ws, mapping))

print(fn(x + 1.0E-9, k))

[0.168 0.201]


In [9]:
# Frequency derivative observable

@jax.jit
def observable(x, k):
    return jax.numpy.log10(1.0E-16 + jax.numpy.linalg.norm(jax.jacrev(fn)(x, k)))

out = observable(x, k)

In [10]:
# Set initial grid in (qx, qy) plane

n = 250

qx = jax.numpy.linspace(0.0, 0.6, n)
qy = jax.numpy.linspace(0.0, 0.6, n)
qs = jax.numpy.stack(jax.numpy.meshgrid(qx, qy, indexing='ij')).swapaxes(-1, 0).reshape(n*n, -1)
ps = jax.numpy.full_like(qs, 1.0E-12)
xs = jax.numpy.hstack([qs, ps])

In [11]:
%%time

# Map

fi = jax.vmap(observable, (0, None))
fs = fi(xs, k).block_until_ready()
fs.shape

CPU times: user 32.4 s, sys: 12.5 s, total: 44.9 s
Wall time: 11.8 s


(62500,)

In [12]:
%%time

# Batched map

xbs = jax.numpy.array_split(xs, n)

xb, *xr = xbs
fi = jax.jit(jax.vmap(observable, (0, None)))
fs = [fi(xb, k)]

for xb in xr:
    fs.append(fi(xb, k))
fs = jax.numpy.concatenate(fs)
fs.shape

CPU times: user 45.6 s, sys: 18.4 s, total: 1min 3s
Wall time: 27.7 s


(62500,)

In [13]:
# Nested derivatives (also vmap over nested)

f0 = fn(x, k).squeeze().block_until_ready()
f1 = jax.jacrev(fn)(x, k).squeeze().block_until_ready()
f2 = jax.jacrev(jax.jacrev(fn))(x, k).squeeze().block_until_ready()

f0.shape, f1.shape, f2.shape

((2,), (2, 4), (2, 4, 4))